# Where Should Additional NHS Elective Capacity Go?
## An Evidence-Based Framework for Reducing Long Waits Under Operational Constraints

**Authors**: Ajmal Baba & Zannix Data Agent  
**Research Entity**: ZannixLabs Applied AI & Healthcare Systems Research  
**Platform**: Zaanix Lakehouse Engine (`http://localhost:4200/mcp`)  
**Dataset**: 1,085,689 longitudinal NHS England RTT records (March 2026 – August 2026)  

---

### 1. Environment Setup & Data Lakehouse Ingestion
We connect directly to the Zaanix Platform lakehouse using Python and DuckDB to access the reconciled analytical tables.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Connect to Zaanix Lakehouse data files
con = duckdb.connect()
print('Connected to DuckDB vectorized analytics engine.')


### 2. The Waiting-List Stock-Flow Accounting Identity
We query the reconciled stock-flow panel from Zaanix Lakehouse to evaluate national inflows, completions, and net residual removals.

In [2]:
df_reconciled = con.execute('''
WITH monthly_panel AS (
    SELECT 
        Period AS period_str,
        SUM(CASE WHEN "RTT Part Description" = 'Incomplete Pathways' THEN "Total All"::BIGINT ELSE 0 END) AS incomplete_stock,
        SUM(CASE WHEN "RTT Part Description" = 'New RTT Periods - All Patients' THEN "Total All"::BIGINT ELSE 0 END) AS clock_starts,
        SUM(CASE WHEN "RTT Part Description" = 'Completed Pathways For Admitted Patients' THEN "Total All"::BIGINT ELSE 0 END) AS admitted_comps,
        SUM(CASE WHEN "RTT Part Description" = 'Completed Pathways For Non-Admitted Patients' THEN "Total All"::BIGINT ELSE 0 END) AS non_admitted_comps
    FROM read_csv_auto('/Users/zaza/Documents/Ajmal-coding-Projects/DuckView/data/2026*-RTT-*.csv', max_line_size=20000000)
    WHERE "Treatment Function Name" = 'Total'
    GROUP BY Period
)
SELECT 
    period_str,
    incomplete_stock,
    clock_starts,
    admitted_comps,
    non_admitted_comps,
    (admitted_comps + non_admitted_comps) AS total_completions,
    LAG(incomplete_stock) OVER (ORDER BY period_str) AS start_stock,
    (LAG(incomplete_stock) OVER (ORDER BY period_str) + clock_starts - (admitted_comps + non_admitted_comps) - incomplete_stock) AS net_residual_removals
FROM monthly_panel;
''').df()
print(df_reconciled.dropna())


### 3. Visualizing the Residual Flow Trap
A comparison between monthly new arrivals, clinical completions, and unrecorded residual removals.

In [3]:
months = ['April', 'May', 'June', 'July', 'August']
starts = [1769633, 1698955, 1937530, 1924841, 1666539]
comps = [1452849, 1406176, 1632188, 1598446, 1387404]
residuals = [203507, 232665, 310937, 227643, 237477]

x = np.arange(len(months))
width = 0.28

plt.figure(figsize=(10, 5), dpi=150)
plt.bar(x - width, [s/1e6 for s in starts], width, label='New Clock Starts (Inflow)', color='#2563EB')
plt.bar(x, [c/1e6 for c in comps], width, label='Recorded Completions (Outflow)', color='#10B981')
plt.bar(x + width, [r/1e6 for r in residuals], width, label='Net Residual Removals (Validation)', color='#F59E0B')

plt.title('NHS England Monthly Elective Inflow, Outflow, and Residual Cleansing (2026)', fontsize=12, fontweight='bold')
plt.ylabel('Million Pathways', fontsize=10)
plt.xticks(x, months)
plt.legend(frameon=True)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### 4. Age-Structured Queue Progression & Policy Optimization
We simulate 6 capacity allocation policies over 12 months under stochastic demand shocks and execution delivery risk.

In [4]:
from src.models.age_structured_queue_simulator import run_policy_benchmark
# Run the complete benchmark and output summary metrics
run_policy_benchmark()


### 5. Policy Benchmark Comparison Table
Evaluating cumulative pathway-weeks >52w and long-tail backlog compression.

In [5]:
policy_summary = pd.DataFrame([
    {'Policy': 'Policy 0 (Status Quo)', 'Mean Cum. PW >52w': 677842, 'Reduction (%)': 0.0, 'Tail >65w': 3005},
    {'Policy': 'Policy 1 (Proportional Queue)', 'Mean Cum. PW >52w': 388264, 'Reduction (%)': 42.72, 'Tail >65w': 1008},
    {'Policy': 'Policy 2 (Long-Wait Burden)', 'Mean Cum. PW >52w': 388378, 'Reduction (%)': 42.70, 'Tail >65w': 1000},
    {'Policy': 'Policy 3 (Net Inflow Deficit)', 'Mean Cum. PW >52w': 409574, 'Reduction (%)': 39.58, 'Tail >65w': 1176},
    {'Policy': 'Policy 4 (Greedy Marginal Benefit)', 'Mean Cum. PW >52w': 386850, 'Reduction (%)': 42.93, 'Tail >65w': 1006},
    {'Policy': 'Policy 5 (Robust Stochastic Min-Max)', 'Mean Cum. PW >52w': 385073, 'Reduction (%)': 43.19, 'Tail >65w': 998},
])
print(policy_summary.to_string(index=False))
